# Dementia QA — Human Evaluation UI (Educational Value)

Rate the quality of automatically generated question–answer pairs from dementia-care videos.

**Data sources** (auto-discovered from the repo):
- `Master/` — 10 videos × 2 approaches (`SingleAgent`, `MultiAgent-LLMChunking`)
- `Teepa/` — 15 videos × 4 approaches (`SingleAgent`, `DualAgent`, `MultiAgent-LLMChunking`, `RAG`)

**How to use:** run all cells top to bottom, then follow the on-screen steps:
1. Choose dataset(s)
2. Choose videos
3. Choose approaches + options (blind mode, shuffle, sampling)
4. Choose metrics, enter your name, and start rating

Progress is checkpointed after every submission, so you can close the notebook and resume later
(same annotator name). Results are saved as an Excel file in `eval/results/`.

Works locally (Jupyter/VS Code) and on Google Colab (mounts Drive — set `COLAB_BASE_DIR` below).

In [1]:
# Install required packages (safe to re-run; mostly needed on Colab / fresh envs)
import importlib.util, subprocess, sys

for pkg in ["ipywidgets", "pandas", "openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\u2713 Packages ready")

✓ Packages ready


In [2]:
# Imports + locate the repository (works locally and on Colab)
import os
import json
import time
import html
import random
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

IN_COLAB = "google.colab" in sys.modules

# If you run this on Colab, upload/sync the repo to Drive and point this at it:
COLAB_BASE_DIR = "/content/drive/MyDrive/MedicalQA"

if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    BASE_DIR = Path(COLAB_BASE_DIR)
else:
    # Notebook lives in eval/ — look for Master/ or Teepa/ in cwd, then parents
    BASE_DIR = Path.cwd()
    for cand in [BASE_DIR, *BASE_DIR.parents]:
        if (cand / "Master").exists() or (cand / "Teepa").exists():
            BASE_DIR = cand
            break

OUTPUT_DIR = BASE_DIR / "eval" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:  {BASE_DIR}")
print(f"Results go: {OUTPUT_DIR}")
assert (BASE_DIR / "Master").exists() or (BASE_DIR / "Teepa").exists(), \
    "Could not find Master/ or Teepa/ - set BASE_DIR manually in this cell."


Repo root:  /Users/ruiweng/Lab/Medicine/MedicalQA
Results go: /Users/ruiweng/Lab/Medicine/MedicalQA/eval/results


In [3]:
# Metric definitions — focused on the EDUCATIONAL value of the dementia QAs
METRIC_DEFS = {
    "Question Fluency":
        "(Grammar, syntax, sentence structure.) The question is grammatically correct "
        "and free from language errors.",
    "Answer Fluency":
        "(Grammar, syntax, sentence structure.) The answer is grammatically correct "
        "and free from language errors.",
    "Question Clarity":
        "(Understandability and specificity.) The question is easy to comprehend and specific "
        "enough to have a clear meaning — not vague or ambiguous.",
    "Answer Clarity":
        "(Understandability.) The answer is easy to comprehend and the explanation is "
        "straightforward — not confusing, rambling, or unclear.",
    "QA-Alignment":
        "(Question\u2013answer correspondence.) The answer directly addresses what the question asks "
        "and adequately satisfies it.",
    "Question Educational Value":
        "(Educational usefulness of the question.) The question targets meaningful knowledge about "
        "dementia or dementia care — a learner (family caregiver, student, or clinician) would gain "
        "useful understanding from having it answered. It is not trivial, overly narrow, or irrelevant.",
    "Answer Educational Value":
        "(Educational quality of the answer.) The answer teaches the reader something meaningful "
        "about dementia or dementia care — it explains concepts, techniques, or practical guidance "
        "accurately and in a way the reader can learn from and apply.",
    "Standalone Quality":
        "(Independence from the source video.) The QA pair makes sense on its own, without having "
        "watched the video — it avoids unresolved references such as \u201cthe transcript\u201d, "
        "\u201cthe speaker\u201d, or \u201cthis segment\u201d that a standalone reader could not follow.",
}

# Which part of the QA pair each metric is judged on (controls highlighting in the UI)
METRIC_TYPES = {
    "q_only":  ["Question Fluency", "Question Clarity", "Question Educational Value"],
    "a_only":  ["Answer Fluency", "Answer Clarity", "Answer Educational Value"],
    "qa_pair": ["QA-Alignment", "Standalone Quality"],
}

ALL_METRICS = [
    "Question Fluency",
    "Answer Fluency",
    "Question Clarity",
    "Answer Clarity",
    "QA-Alignment",
    "Question Educational Value",
    "Answer Educational Value",
    "Standalone Quality",
]

LIKERT_OPTIONS = [
    ("1 - Strongly Disagree", 1),
    ("2 - Disagree", 2),
    ("3 - Neutral", 3),
    ("4 - Agree", 4),
    ("5 - Strongly Agree", 5),
]

print(f"\u2713 {len(ALL_METRICS)} metrics configured")

✓ 8 metrics configured


In [4]:
# Load every finalQA.json under Master/ and Teepa/
DATASETS = ["Master", "Teepa"]

all_qa_data = []      # every QA pair found on disk
filtered_qa_data = [] # the subset the annotator will rate
current_qa_index = 0
selected_datasets = []
selected_videos = []
selected_approaches = []
selected_metrics = ALL_METRICS[:]
annotator_name = ""
blind_mode = True


def load_all_qa():
    """Discover and load all QA files: <dataset>/<video>/<approach>/QA results/finalQA.json"""
    global all_qa_data
    all_qa_data = []
    for dataset in DATASETS:
        droot = BASE_DIR / dataset
        if not droot.exists():
            continue
        video_dirs = sorted(
            [d for d in droot.iterdir() if d.is_dir() and d.name.isdigit()],
            key=lambda d: int(d.name),
        )
        for vdir in video_dirs:
            for adir in sorted(d for d in vdir.iterdir() if d.is_dir()):
                qa_file = adir / "QA results" / "finalQA.json"
                if not qa_file.exists():
                    continue
                with open(qa_file, encoding="utf-8") as f:
                    items = json.load(f)
                for i, item in enumerate(items):
                    all_qa_data.append({
                        "uid": f"{dataset}_v{vdir.name}_{adir.name}_q{i + 1}",
                        "dataset": dataset,
                        "video": int(vdir.name),
                        "approach": adir.name,
                        "qa_num": i + 1,
                        "question": str(item.get("question", "")).strip(),
                        "answer": str(item.get("answer", "")).strip(),
                        "scores": {},
                    })

    # Summary
    print(f"\u2713 Loaded {len(all_qa_data)} QA pairs total\n")
    summary = (
        pd.DataFrame(all_qa_data)
        .groupby(["dataset", "approach"])
        .agg(videos=("video", "nunique"), qa_pairs=("uid", "count"))
    )
    display(summary)


load_all_qa()

✓ Loaded 1587 QA pairs total



videos  qa_pairs
dataset approach                                
Master  MultiAgent-LLMChunking      10       200
        SingleAgent                 10       199
Teepa   DualAgent                   15       300
        MultiAgent-LLMChunking      15       300
        RAG                         15       300
        SingleAgent                 15       288

In [5]:
# Checkpointing — progress survives kernel restarts / closed tabs
def _checkpoint_path():
    safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in (annotator_name or "anonymous"))
    return OUTPUT_DIR / f"checkpoint_{safe}.json"


def save_checkpoint():
    data = {qa["uid"]: qa["scores"] for qa in filtered_qa_data if qa["scores"]}
    with open(_checkpoint_path(), "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)


def restore_checkpoint():
    """Merge previously saved scores into the current selection. Returns #restored."""
    path = _checkpoint_path()
    if not path.exists():
        return 0
    with open(path, encoding="utf-8") as f:
        saved = json.load(f)
    n = 0
    for qa in filtered_qa_data:
        if qa["uid"] in saved and saved[qa["uid"]]:
            qa["scores"].update(saved[qa["uid"]])
            n += 1
    return n


def fmt_text(text):
    """Escape HTML and preserve line breaks for display."""
    return html.escape(text).replace("\n", "<br>")

print("\u2713 Helpers ready")

✓ Helpers ready


In [6]:
# Step 1 — dataset selection
def show_dataset_selection():
    clear_output(wait=True)
    print("=" * 80)
    print("DEMENTIA QA EVALUATION \u2014 EDUCATIONAL VALUE")
    print("=" * 80)
    print("\nStep 1/4: Choose dataset(s)\n")

    available = sorted({qa["dataset"] for qa in all_qa_data})
    options = [[d] for d in available]
    if len(available) > 1:
        options.append(available)  # "Both"

    def on_click(button):
        global selected_datasets
        selected_datasets = button._datasets
        show_video_selection()

    buttons = []
    for opt in options:
        label = " + ".join(opt) if len(opt) > 1 else opt[0]
        n = sum(1 for qa in all_qa_data if qa["dataset"] in opt)
        btn = widgets.Button(
            description=f"{label}  ({n} QAs)",
            button_style="info",
            layout=widgets.Layout(width="240px", height="50px"),
        )
        btn._datasets = opt
        btn.on_click(on_click)
        buttons.append(btn)

    display(widgets.HBox(buttons))

In [7]:
# Step 2 — video selection
def show_video_selection():
    clear_output(wait=True)
    print("=" * 80)
    print(f"Dataset(s): {', '.join(selected_datasets)}")
    print("=" * 80)
    print("\nStep 2/4: Select videos (all selected by default)\n")

    pool = [qa for qa in all_qa_data if qa["dataset"] in selected_datasets]
    video_keys = sorted({(qa["dataset"], qa["video"]) for qa in pool})

    checkboxes = []
    for ds, vid in video_keys:
        n = sum(1 for qa in pool if qa["dataset"] == ds and qa["video"] == vid)
        cb = widgets.Checkbox(
            value=True,
            description=f"{ds} \u2014 video {vid} ({n})",
            indent=False,
            layout=widgets.Layout(width="240px"),
        )
        cb._key = (ds, vid)
        checkboxes.append(cb)

    def set_all(value):
        for cb in checkboxes:
            cb.value = value

    btn_all = widgets.Button(description="Select all", layout=widgets.Layout(width="110px"))
    btn_none = widgets.Button(description="Select none", layout=widgets.Layout(width="110px"))
    btn_all.on_click(lambda b: set_all(True))
    btn_none.on_click(lambda b: set_all(False))

    next_button = widgets.Button(
        description="Next: Approaches \u2192",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px"),
    )
    status = widgets.HTML("<i>Select at least one video and click Next</i>")

    def on_next(b):
        global selected_videos
        selected_videos = [cb._key for cb in checkboxes if cb.value]
        if not selected_videos:
            status.value = "<span style='color:red;'>\u26a0 Please select at least one video</span>"
            return
        show_approach_selection()

    next_button.on_click(on_next)

    grid = widgets.GridBox(
        checkboxes,
        layout=widgets.Layout(grid_template_columns="repeat(3, 260px)"),
    )
    display(widgets.VBox([
        widgets.HBox([btn_all, btn_none]),
        grid,
        next_button,
        status,
    ]))

In [8]:
# Step 3 — approach selection + options (blind mode, shuffle, sampling)
def show_approach_selection():
    clear_output(wait=True)
    print("=" * 80)
    print(f"Dataset(s): {', '.join(selected_datasets)} | Videos: {len(selected_videos)}")
    print("=" * 80)
    print("\nStep 3/4: Select approaches + options\n")

    pool = [qa for qa in all_qa_data
            if (qa["dataset"], qa["video"]) in selected_videos]
    approaches = sorted({qa["approach"] for qa in pool})

    checkboxes = []
    for app in approaches:
        n = sum(1 for qa in pool if qa["approach"] == app)
        cb = widgets.Checkbox(
            value=True,
            description=f"{app} ({n} QAs)",
            indent=False,
            layout=widgets.Layout(width="320px"),
        )
        cb._approach = app
        checkboxes.append(cb)

    blind_cb = widgets.Checkbox(
        value=True, indent=False,
        description="Blind mode \u2014 hide approach names while rating (recommended)",
        layout=widgets.Layout(width="500px"),
    )
    shuffle_cb = widgets.Checkbox(
        value=True, indent=False,
        description="Shuffle QA order (recommended, avoids order bias)",
        layout=widgets.Layout(width="500px"),
    )
    seed_input = widgets.IntText(value=42, description="Seed:", layout=widgets.Layout(width="180px"))
    sample_input = widgets.IntText(
        value=0, description="Sample:", layout=widgets.Layout(width="180px"),
    )
    sample_note = widgets.HTML(
        "<i>QAs per approach per video (0 = all). Sampling uses the seed, so all annotators "
        "with the same seed rate the same subset.</i>"
    )

    next_button = widgets.Button(
        description="Next: Metrics \u2192",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px"),
    )
    status = widgets.HTML("<i>Select at least one approach and click Next</i>")

    def on_next(b):
        global selected_approaches, filtered_qa_data, blind_mode
        selected_approaches = [cb._approach for cb in checkboxes if cb.value]
        if not selected_approaches:
            status.value = "<span style='color:red;'>\u26a0 Please select at least one approach</span>"
            return
        blind_mode = blind_cb.value
        seed = seed_input.value
        per_group = max(0, sample_input.value)

        chosen = [qa for qa in pool if qa["approach"] in selected_approaches]

        # Deterministic sampling per (dataset, video, approach) group
        if per_group:
            rng = random.Random(seed)
            groups = {}
            for qa in chosen:
                groups.setdefault((qa["dataset"], qa["video"], qa["approach"]), []).append(qa)
            chosen = []
            for key in sorted(groups):
                items = groups[key]
                if len(items) > per_group:
                    items = rng.sample(items, per_group)
                    items.sort(key=lambda q: q["qa_num"])
                chosen.extend(items)

        if shuffle_cb.value:
            random.Random(seed).shuffle(chosen)
        else:
            chosen.sort(key=lambda q: (q["dataset"], q["video"], q["qa_num"], q["approach"]))

        filtered_qa_data = chosen
        show_metric_selection()

    next_button.on_click(on_next)

    display(widgets.VBox([
        widgets.HTML("<b>Approaches:</b>"),
        widgets.VBox(checkboxes),
        widgets.HTML("<br><b>Options:</b>"),
        blind_cb,
        shuffle_cb,
        widgets.HBox([seed_input, sample_input]),
        sample_note,
        widgets.HTML("<br>"),
        next_button,
        status,
    ]))

In [9]:
# Step 4 — metric selection + annotator name
def show_metric_selection():
    clear_output(wait=True)
    print("=" * 80)
    print(f"Dataset(s): {', '.join(selected_datasets)} | Videos: {len(selected_videos)} "
          f"| Approaches: {', '.join(selected_approaches)}")
    print(f"QA pairs to evaluate: {len(filtered_qa_data)}")
    print("=" * 80)
    print("\nStep 4/4: Select metrics and enter your name\n")

    checkboxes = []
    for metric in ALL_METRICS:
        cb = widgets.Checkbox(
            value=True, description=metric, indent=False,
            layout=widgets.Layout(width="320px"),
        )
        checkboxes.append(cb)

    name_input = widgets.Text(
        value="", placeholder="Enter your name",
        description="Annotator:", layout=widgets.Layout(width="400px"),
    )
    start_button = widgets.Button(
        description="\u2713 Start Evaluation",
        button_style="success",
        layout=widgets.Layout(width="200px", height="40px"),
    )
    status = widgets.HTML("<i>Select metrics and click Start</i>")

    def on_start(b):
        global selected_metrics, current_qa_index, annotator_name
        selected_metrics = [cb.description for cb in checkboxes if cb.value]
        annotator_name = name_input.value.strip()
        if not selected_metrics:
            status.value = "<span style='color:red;'>\u26a0 Please select at least one metric</span>"
            return

        restored = restore_checkpoint()
        # Jump to the first QA that still has unrated selected metrics
        current_qa_index = 0
        for i, qa in enumerate(filtered_qa_data):
            if not all(m in qa["scores"] for m in selected_metrics):
                current_qa_index = i
                break
        else:
            current_qa_index = len(filtered_qa_data)

        if restored:
            print(f"\n\u2713 Resumed checkpoint: {restored} QA pairs already have scores; "
                  f"continuing at QA {current_qa_index + 1}")
            time.sleep(1.5)
        show_qa_evaluation()

    start_button.on_click(on_start)

    display(widgets.VBox([
        widgets.HTML("<b>Metrics to evaluate (all selected by default):</b>"),
        widgets.VBox(checkboxes),
        widgets.HTML("<br>"),
        name_input,
        start_button,
        status,
    ]))

In [10]:
# Evaluation screen
def show_qa_evaluation():
    global current_qa_index

    if current_qa_index >= len(filtered_qa_data):
        save_results()
        return

    clear_output(wait=True)
    qa = filtered_qa_data[current_qa_index]
    total = len(filtered_qa_data)
    done = sum(1 for q in filtered_qa_data
               if all(m in q["scores"] for m in selected_metrics))

    approach_label = "(hidden \u2014 blind mode)" if blind_mode else qa["approach"]
    print("=" * 80)
    print(f"QA {current_qa_index + 1} of {total}   |   fully rated so far: {done}")
    print("=" * 80)
    print(f"Dataset: {qa['dataset']} | Video: {qa['video']} | Approach: {approach_label}")
    print("=" * 80)

    # The QA pair, shown once at the top
    display(HTML(f"""
    <div style="background:#fff3cd; padding:14px; margin:10px 0; border-radius:8px;
                border-left:4px solid #ff9800;">
        <strong style="color:#856404;">Question</strong><br>
        <span style="color:#333; font-size:15px;">{fmt_text(qa['question'])}</span>
    </div>
    <div style="background:#d4edda; padding:14px; margin:10px 0; border-radius:8px;
                border-left:4px solid #28a745; max-height:400px; overflow-y:auto;">
        <strong style="color:#155724;">Answer</strong><br>
        <span style="color:#333; font-size:15px;">{fmt_text(qa['answer'])}</span>
    </div>
    """))

    tag_colors = {"q_only": ("#856404", "Question"), "a_only": ("#155724", "Answer"),
                  "qa_pair": ("#2c5aa0", "Q&A pair")}

    radio_buttons = {}
    metric_rows = []
    for metric in selected_metrics:
        mtype = next(t for t, ms in METRIC_TYPES.items() if metric in ms)
        color, tag = tag_colors[mtype]
        header = widgets.HTML(f"""
        <div style="background:#f5f5f5; padding:10px 14px; margin-top:14px; border-radius:8px;
                    border-left:4px solid #4da6ff;">
            <b style="color:#2c5aa0; font-size:15px;">{metric}</b>
            <span style="background:{color}; color:white; border-radius:10px; padding:1px 8px;
                         font-size:11px; margin-left:8px;">{tag}</span><br>
            <span style="color:#555; font-style:italic; font-size:13px;">{METRIC_DEFS[metric]}</span>
        </div>""")
        radio = widgets.RadioButtons(
            options=LIKERT_OPTIONS,
            value=qa["scores"].get(metric),  # None until the annotator chooses
            layout=widgets.Layout(width="600px", margin="4px 0 0 10px"),
        )
        radio_buttons[metric] = radio
        metric_rows += [header, radio]

    btn_prev = widgets.Button(
        description="\u2190 Previous", button_style="info",
        layout=widgets.Layout(width="150px", height="40px"),
        disabled=(current_qa_index == 0),
    )
    btn_next = widgets.Button(
        description="Submit & Next \u2192", button_style="success",
        layout=widgets.Layout(width="180px", height="40px"),
    )
    btn_save = widgets.Button(
        description="\U0001f4be Save & Exit", button_style="warning",
        layout=widgets.Layout(width="150px", height="40px"),
    )
    status = widgets.HTML("")

    def store_scores():
        for metric, radio in radio_buttons.items():
            if radio.value is not None:
                qa["scores"][metric] = radio.value
        save_checkpoint()

    def on_next(b):
        global current_qa_index
        missing = [m for m, r in radio_buttons.items() if r.value is None]
        if missing:
            status.value = ("<span style='color:red;'>\u26a0 Please rate: "
                            + ", ".join(missing) + "</span>")
            return
        store_scores()
        current_qa_index += 1
        show_qa_evaluation()

    def on_prev(b):
        global current_qa_index
        store_scores()
        current_qa_index -= 1
        show_qa_evaluation()

    def on_save(b):
        store_scores()
        save_results(partial=True)

    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)
    btn_save.on_click(on_save)

    display(widgets.VBox(metric_rows))
    display(widgets.HBox([btn_prev, btn_next, btn_save],
                         layout=widgets.Layout(margin="20px 0")))
    display(status)

In [11]:
# Save results to Excel
def save_results(partial=False):
    clear_output(wait=True)
    print("=" * 80)
    print("SAVING RESULTS" + (" (partial \u2014 you can resume later)" if partial else ""))
    print("=" * 80)

    rows = []
    for qa in filtered_qa_data:
        row = {
            "Dataset": qa["dataset"],
            "Video Index": qa["video"],
            "Approach": qa["approach"],
            "QA ID": qa["uid"],
            "Question": qa["question"],
            "Answer": qa["answer"],
        }
        for metric in ALL_METRICS:
            row[metric] = qa["scores"].get(metric, "")
        row["Annotator"] = annotator_name
        rows.append(row)

    df = pd.DataFrame(rows)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    name_part = annotator_name if annotator_name else "anonymous"
    ds_part = "-".join(selected_datasets)
    filename = f"{ds_part}_{name_part}_Eval_{timestamp}.xlsx"
    out_path = OUTPUT_DIR / filename
    df.to_excel(out_path, index=False, engine="openpyxl")

    rated = sum(1 for qa in filtered_qa_data
                if all(m in qa["scores"] for m in selected_metrics))
    print(f"\n\u2713 Saved to: {out_path}")
    print(f"\u2713 QA pairs fully rated: {rated} / {len(filtered_qa_data)}")
    print(f"\u2713 Annotator: {annotator_name or '(anonymous)'}")
    print(f"\u2713 Metrics: {', '.join(selected_metrics)}")
    if partial:
        print("\n\u23f8 Progress is checkpointed. To resume: re-run the start cell, make the "
              "same selections, and enter the same annotator name.")
    else:
        print("\n\U0001f389 Evaluation complete \u2014 thank you!")

    if IN_COLAB:
        try:
            files.download(str(out_path))
            print("\u2713 Download started")
        except Exception as e:
            print(f"\u26a0 Auto-download failed ({e}); download it from the Files panel.")

    # Mean score per approach (only over rated pairs)
    scored = df[df[selected_metrics[0]] != ""] if selected_metrics else df
    if len(scored):
        print("\n" + "=" * 80)
        print("MEAN SCORE PER APPROACH (rated pairs only)")
        print("=" * 80)
        display(scored.groupby("Approach")[selected_metrics]
                      .apply(lambda g: g.apply(pd.to_numeric, errors="coerce").mean())
                      .round(2))

    display(df.head(10))

In [12]:
# \u25b6 START — run this cell to begin (re-run it to start over)
show_dataset_selection()

,Question Fluency,Answer Fluency,Question Clarity,Answer Clarity,QA-Alignment,Question Educational Value,Answer Educational Value,Standalone Quality
Approach,,,,,,,,
MultiAgent-LLMChunking,5.0,5.0,5.0,5.0,5.0,2.0,4.0,4.0


,Dataset,Video Index,Approach,QA ID,Question,Answer,Question Fluency,Answer Fluency,Question Clarity,Answer Clarity,QA-Alignment,Question Educational Value,Answer Educational Value,Standalone Quality,Annotator
0,Master,2,MultiAgent-LLMChunking,Master_v2_MultiAgent-LLMChunking_q6,How does the speaker describe the approach to ...,The speaker describes the approach to role-pla...,5,5,5,5,5,2,4,4,Ruiwen
1,Master,3,SingleAgent,Master_v3_SingleAgent_q11,How can a caregiver prevent a person from slid...,A caregiver can physically block the person's ...,,,,,,,,,Ruiwen
2,Master,1,MultiAgent-LLMChunking,Master_v1_MultiAgent-LLMChunking_q10,"What is the significance of the ""bell"" notific...","The ""bell"" notification feature is significant...",,,,,,,,,Ruiwen
3,Teepa,1,SingleAgent,Teepa_v1_SingleAgent_q6,How does the deterioration of the sensory-moto...,The brain contains a map for sensing the body ...,,,,,,,,,Ruiwen
4,Master,3,SingleAgent,Master_v3_SingleAgent_q8,How should the person's head be positioned dur...,The person's head should be allowed to rest ag...,,,,,,,,,Ruiwen
5,Master,2,SingleAgent,Master_v2_SingleAgent_q15,What specific technique is suggested for handl...,"Instead of asking ""Where have you been?"" which...",,,,,,,,,Ruiwen
6,Teepa,3,SingleAgent,Teepa_v3_SingleAgent_q10,How did the caregiver create a motivating reas...,The caregiver framed the task as a personal re...,,,,,,,,,Ruiwen
7,Master,2,MultiAgent-LLMChunking,Master_v2_MultiAgent-LLMChunking_q14,How does the example of misidentifying weather...,The example of misidentifying weather conditio...,,,,,,,,,Ruiwen
8,Teepa,3,MultiAgent-LLMChunking,Teepa_v3_MultiAgent-LLMChunking_q15,"In the context of collaborative learning, how ...","Based on the provided transcript, the instruct...",,,,,,,,,Ruiwen
9,Master,3,SingleAgent,Master_v3_SingleAgent_q15,In what other situations besides bed-to-chair ...,"These techniques, particularly the use of grav...",,,,,,,,,Ruiwen


---
## Optional: aggregate results across annotators

Run the cell below after collecting one or more result files in `eval/results/` to see mean
scores per approach and per metric across all annotators.

In [ ]:
result_files = sorted(OUTPUT_DIR.glob("*_Eval_*.xlsx"))
if not result_files:
    print("No result files found in", OUTPUT_DIR)
else:
    frames = []
    for f in result_files:
        d = pd.read_excel(f)
        d["__file"] = f.name
        frames.append(d)
    combined = pd.concat(frames, ignore_index=True)
    metric_cols = [m for m in ALL_METRICS if m in combined.columns]
    combined[metric_cols] = combined[metric_cols].apply(pd.to_numeric, errors="coerce")
    rated = combined.dropna(subset=metric_cols, how="all")

    print(f"Files: {len(result_files)} | Rated rows: {len(rated)} | "
          f"Annotators: {rated['Annotator'].nunique()}")

    print("\nMean score per approach (all datasets):")
    display(rated.groupby("Approach")[metric_cols].mean().round(2))

    print("\nMean score per dataset + approach:")
    display(rated.groupby(["Dataset", "Approach"])[metric_cols].mean().round(2))